Cell 1 – Khởi tạo và đọc dữ liệu

In [6]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import json
import warnings
import os

warnings.filterwarnings("ignore")

CURRENT_DIR = Path.cwd().resolve()
candidates = [CURRENT_DIR, *CURRENT_DIR.parents]

for env_name in ("VSCODE_CWD", "GITHUB_WORKSPACE"):
    env_dir = os.environ.get(env_name)
    if env_dir:
        candidates.append(Path(env_dir).resolve())

PROJECT_DIR = next(
    (candidate for candidate in candidates if (candidate / "data" / "raw").is_dir()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục project chứa data/raw/. "
        "Hãy mở VS Code tại thư mục SKU-Weekly-Sales-Forecasting "
        "hoặc đặt kernel working directory về thư mục project. "
        f"Thư mục hiện tại: {CURRENT_DIR}"
    )

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_safely(file_path):
    for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False
            )
        except Exception:
            pass
    raise ValueError(f"Không đọc được {file_path}")


tables = {
    file.stem: read_csv_safely(file)
    for file in sorted(RAW_DIR.glob("*.csv"))
}

if not tables:
    raise FileNotFoundError("Không tìm thấy dữ liệu trong data/raw/.")

print("Đã đọc", len(tables), "bảng")

Đã đọc 27 bảng


Cell 2 – Khai báo cấu hình JOIN

In [7]:
for table_name, df in tables.items():
    print("\n", "=" * 80)
    print("TABLE:", table_name)
    print("ROWS:", len(df))
    print("COLUMNS:", df.columns.tolist())


# Ví dụ cấu trúc cấu hình.
# Chỉ bỏ comment và sửa tên cột sau khi xác nhận trong dữ liệu thực tế.
#
JOIN_CONFIG = [
    {
        "left_table": "SalesOrderLines",
        "right_table": "SalesOrders",
        "left_key": "SalesOrderID",
        "right_key": "SalesOrderID",
        "how": "left",
        "validate": "many_to_one"
    },
    {
        "left_table": "SalesOrderLines",
        "right_table": "ProductVariants",
        "left_key": "SKU_ID",
        "right_key": "SKU_ID",
        "how": "left",
        "validate": "many_to_one"
    },
    {
        "left_table": "ProductVariants",
        "right_table": "Products",
        "left_key": "ProductID",
        "right_key": "ProductID",
        "how": "left",
        "validate": "many_to_one"
    }
]

print(
    "JOIN_CONFIG hiện có", len(JOIN_CONFIG),
    "quan hệ. Nếu chưa xác nhận khóa, notebook sẽ không tự JOIN."
)


TABLE: Brands
ROWS: 66
COLUMNS: ['BrandID', 'BrandName']

TABLE: BrandSubcategoryMap
ROWS: 310
COLUMNS: ['BrandID', 'SubcategoryID']

TABLE: Categories
ROWS: 8
COLUMNS: ['CategoryID', 'CategoryName']

TABLE: CustomerReturns
ROWS: 23790
COLUMNS: ['CustomerReturnID', 'SalesOrderLineID', 'ReturnDate', 'ReturnQuantity', 'ReturnReason', 'ReturnOutcome', 'ReturnedCondition', 'ReturnLocationID', 'RefundAmount', 'RefundDate', 'RefundStatus']

TABLE: Customers
ROWS: 140368
COLUMNS: ['CustomerID', 'CustomerName', 'HomeGovernorateID', 'AgeBand', 'CustomerSinceDate']

TABLE: Governorates
ROWS: 27
COLUMNS: ['GovernorateID', 'GovernorateName']

TABLE: InventoryBalances
ROWS: 12104
COLUMNS: ['SKU_ID', 'LocationID', 'AvailableQuantity', 'NonSellableQuantity', 'WeightedAvgUnitCost', 'ReorderPointQty', 'SafetyStockQty', 'MinStockQty', 'MaxStockQty']

TABLE: InventoryMovements
ROWS: 961048
COLUMNS: ['InventoryMovementID', 'MovementDateTime', 'SKU_ID', 'MovementType', 'Quantity', 'FromLocationID', 'ToLoc

Cell 3 – Chuẩn hóa kiểu dữ liệu và tên trạng thái

In [8]:
cleaned_tables = {}
preprocessing_log = []

for table_name, original_df in tables.items():

    df = original_df.copy()

    rows_before = len(df)
    columns_before = len(df.columns)

    # Chuẩn hóa khoảng trắng trong tên cột.
    df.columns = [
        re.sub(r"\s+", "_", str(col).strip())
        for col in df.columns
    ]

    # Chuẩn hóa chuỗi: bỏ khoảng trắng thừa, giữ nguyên nội dung.
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(
            lambda value: (
                value.strip() if isinstance(value, str) else value
            )
        )

    # Chuẩn hóa trạng thái đơn hàng nếu tìm thấy cột phù hợp.
    status_candidates = [
        col for col in df.columns
        if "status" in col.lower()
    ]

    for status_col in status_candidates:
        df[status_col] = (
            df[status_col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

    # Chuyển các cột có tên gợi ý ngày tháng sang datetime.
    date_candidates = [
        col for col in df.columns
        if any(
            token in col.lower()
            for token in ["date", "datetime", "timestamp", "_at", "time"]
        )
    ]

    for col in date_candidates:
        converted = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        # Chỉ chuyển nếu phần lớn giá trị không thiếu ban đầu
        # chuyển đổi thành ngày thành công.
        original_present = df[col].notna().sum()

        if original_present > 0:
            success_rate = converted.notna().sum() / original_present

            if success_rate >= 0.7:
                df[col] = converted

    cleaned_tables[table_name] = df

    preprocessing_log.append({
        "table": table_name,
        "rows_before": rows_before,
        "rows_after": len(df),
        "columns_before": columns_before,
        "columns_after": len(df.columns),
        "operation": "Trim strings, normalize status, parse date candidates"
    })

print("Đã chuẩn hóa", len(cleaned_tables), "bảng.")

Đã chuẩn hóa 27 bảng.


Cell 4 – Kiểm tra trùng lặp và Quantity âm

In [9]:
duplicate_tables = {}
negative_quantity_tables = {}

for table_name, df in cleaned_tables.items():

    exact_duplicate_mask = df.duplicated(keep="first")
    exact_duplicates = df.loc[exact_duplicate_mask].copy()

    duplicate_tables[table_name] = exact_duplicates

    # Chỉ loại bản ghi trùng hoàn toàn.
    cleaned_tables[table_name] = (
        df.loc[~exact_duplicate_mask].copy()
    )

    # Tìm cột Quantity theo tên.
    quantity_col = next(
        (
            col for col in cleaned_tables[table_name].columns
            if col.lower() in [
                "quantity", "qty", "salesquantity", "unitssold"
            ]
        ),
        None
    )

    if quantity_col:
        qty = pd.to_numeric(
            cleaned_tables[table_name][quantity_col],
            errors="coerce"
        )

        negative_mask = qty < 0

        negative_quantity_tables[table_name] = (
            cleaned_tables[table_name].loc[negative_mask].copy()
        )

        print(
            table_name,
            "| Exact duplicates removed:", int(exact_duplicate_mask.sum()),
            "| Negative Quantity:", int(negative_mask.sum())
        )

    else:
        negative_quantity_tables[table_name] = pd.DataFrame()

print("Đã hoàn thành kiểm tra trùng lặp và Quantity âm.")

InventoryMovements | Exact duplicates removed: 0 | Negative Quantity: 0
SalesOrderLines | Exact duplicates removed: 0 | Negative Quantity: 0
Đã hoàn thành kiểm tra trùng lặp và Quantity âm.


Cell 5 – JOIN theo cấu hình đã xác nhận

In [10]:
joined_df = None
join_log = []

if JOIN_CONFIG:

    first_config = JOIN_CONFIG[0]
    base_table = first_config["left_table"]

    if base_table not in cleaned_tables:
        raise KeyError(f"Không tìm thấy bảng {base_table}")

    joined_df = cleaned_tables[base_table].copy()

    for config in JOIN_CONFIG:

        left_table = config["left_table"]
        right_table = config["right_table"]

        left_key = config["left_key"]
        right_key = config["right_key"]

        how = config.get("how", "left")
        validate = config.get("validate", "many_to_one")

        if right_table not in cleaned_tables:
            raise KeyError(f"Không tìm thấy bảng {right_table}")

        right_df = cleaned_tables[right_table].copy()

        if left_key not in joined_df.columns:
            raise KeyError(
                f"Không tìm thấy khóa trái {left_key} trong bảng đang JOIN."
            )

        if right_key not in right_df.columns:
            raise KeyError(
                f"Không tìm thấy khóa phải {right_key} trong {right_table}."
            )

        rows_before = len(joined_df)

        joined_df = joined_df.merge(
            right_df,
            left_on=left_key,
            right_on=right_key,
            how=how,
            validate=validate,
            suffixes=("", f"_{right_table}")
        )

        join_log.append({
            "left_table": left_table,
            "right_table": right_table,
            "left_key": left_key,
            "right_key": right_key,
            "join_type": how,
            "validate": validate,
            "rows_before": rows_before,
            "rows_after": len(joined_df)
        })

    print("JOIN hoàn thành:", joined_df.shape)

else:
    print(
        "Chưa có JOIN_CONFIG đã xác nhận. "
        "Các bảng được giữ riêng, không tự ghép bằng khóa phỏng đoán."
    )

# Chạy pipeline tiền xử lý có kiểm soát để sinh report thiếu dữ liệu,
# duplicate, ngoại lai IQR, transaction hợp lệ và JOIN cuối cùng.
import subprocess
import sys

preprocess_script = PROJECT_DIR / "src" / "data" / "preprocess_lab.py"
subprocess.run([
    sys.executable,
    str(preprocess_script),
    "--raw-dir", str(PROJECT_DIR / "data" / "raw"),
    "--processed-dir", str(PROJECT_DIR / "data" / "processed"),
    "--report-dir", str(PROJECT_DIR / "reports" / "tables")
], check=True)


JOIN hoàn thành: (579572, 31)


Cell 6 – Validation sau xử lý

In [11]:
validation_rows = []

for table_name, df in cleaned_tables.items():

    validation_rows.append({
        "table": table_name,
        "check": "Row count",
        "value": len(df),
        "status": "PASS" if len(df) > 0 else "WARNING",
        "detail": "Số dòng sau xử lý"
    })

    validation_rows.append({
        "table": table_name,
        "check": "Exact duplicate rows",
        "value": int(df.duplicated().sum()),
        "status": (
            "PASS" if df.duplicated().sum() == 0 else "WARNING"
        ),
        "detail": "Dòng trùng hoàn toàn còn lại"
    })

    # Kiểm tra ngày tương lai nếu có cột datetime.
    date_cols = df.select_dtypes(
        include=["datetime", "datetimetz"]
    ).columns

    for col in date_cols:
        future_count = int(
            (df[col] > pd.Timestamp.today()).sum()
        )

        validation_rows.append({
            "table": table_name,
            "check": f"Future date - {col}",
            "value": future_count,
            "status": "PASS" if future_count == 0 else "WARNING",
            "detail": "Ngày lớn hơn ngày chạy notebook"
        })


if joined_df is not None:
    validation_rows.append({
        "table": "JOINED_DATA",
        "check": "Joined row count",
        "value": len(joined_df),
        "status": "PASS" if len(joined_df) > 0 else "WARNING",
        "detail": "Số dòng sau JOIN"
    })


validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,table,check,value,status,detail
0,Brands,Row count,66,PASS,Số dòng sau xử lý
1,Brands,Exact duplicate rows,0,PASS,Dòng trùng hoàn toàn còn lại
2,BrandSubcategoryMap,Row count,310,PASS,Số dòng sau xử lý
3,BrandSubcategoryMap,Exact duplicate rows,0,PASS,Dòng trùng hoàn toàn còn lại
4,Categories,Row count,8,PASS,Số dòng sau xử lý
...,...,...,...,...,...
64,Subcategories,Row count,65,PASS,Số dòng sau xử lý
65,Subcategories,Exact duplicate rows,0,PASS,Dòng trùng hoàn toàn còn lại
66,Suppliers,Row count,16,PASS,Số dòng sau xử lý
67,Suppliers,Exact duplicate rows,0,PASS,Dòng trùng hoàn toàn còn lại


Cell 7 – Xuất dữ liệu đã xử lý và bảng cách ly

In [12]:
# Xuất từng bảng sau xử lý.
for table_name, df in cleaned_tables.items():
    output_path = PROCESSED_DIR / f"{table_name}_cleaned.csv"

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

# Xuất bảng đã JOIN nếu có.
if joined_df is not None:
    joined_df.to_csv(
        PROCESSED_DIR / "joined_sales_data.csv",
        index=False,
        encoding="utf-8-sig"
    )

# Xuất các dòng trùng hoàn toàn đã loại.
duplicate_dir = PROCESSED_DIR / "quarantine" / "duplicates"
duplicate_dir.mkdir(parents=True, exist_ok=True)

for table_name, df in duplicate_tables.items():
    if not df.empty:
        df.to_csv(
            duplicate_dir / f"{table_name}_duplicates.csv",
            index=False,
            encoding="utf-8-sig"
        )

# Xuất Quantity âm để điều tra, không tự sửa hoặc chuyển thành 0.
negative_dir = PROCESSED_DIR / "quarantine" / "negative_quantity"
negative_dir.mkdir(parents=True, exist_ok=True)

for table_name, df in negative_quantity_tables.items():
    if not df.empty:
        df.to_csv(
            negative_dir / f"{table_name}_negative_quantity.csv",
            index=False,
            encoding="utf-8-sig"
        )

print("Đã xuất dữ liệu vào:", PROCESSED_DIR)

Đã xuất dữ liệu vào: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\data\processed


Cell 8 – Xuất preprocessing_report.xlsx

In [13]:
preprocessing_summary = []

for table_name, original_df in tables.items():
    final_df = cleaned_tables[table_name]

    preprocessing_summary.append({
        "table": table_name,
        "rows_before": len(original_df),
        "rows_after": len(final_df),
        "rows_removed_exact_duplicates": (
            len(original_df) - len(final_df)
        ),
        "columns_before": len(original_df.columns),
        "columns_after": len(final_df.columns),
        "missing_cells_before": int(original_df.isna().sum().sum()),
        "missing_cells_after": int(final_df.isna().sum().sum())
    })

preprocessing_summary_df = pd.DataFrame(
    preprocessing_summary
)

join_log_df = pd.DataFrame(join_log)

preprocessing_report_path = (
    REPORT_DIR / "preprocessing_report.xlsx"
)

with pd.ExcelWriter(
    preprocessing_report_path,
    engine="openpyxl"
) as writer:

    preprocessing_summary_df.to_excel(
        writer,
        sheet_name="Before After Summary",
        index=False
    )

    validation_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

    join_log_df.to_excel(
        writer,
        sheet_name="Join Log",
        index=False
    )

    pd.DataFrame(preprocessing_log).to_excel(
        writer,
        sheet_name="Preprocessing Log",
        index=False
    )

print("Đã xuất:", preprocessing_report_path)

Đã xuất: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\reports\preprocessing_report.xlsx
